In [6]:
from dotenv import load_dotenv
import os

load_dotenv()  # take environment variables from .env.

if os.environ['OPENAI_API_KEY']:
    print("Open API Key is set")
else:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")

Open API Key is set


In [7]:
# taken from https://docs.langchain.com/oss/python/integrations/chat/openai - instantiation section
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-5.4-nano-2026-03-17")

## **TOOLS**

### **Tavily**

In [44]:
import getpass
import os
from langchain_tavily import TavilySearch

if os.environ.get("TAVILY_API_KEY"):
    print("Tavily API Key is set")
else:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

@tool
def tavily_search(query:str) -> str:
    """This tool allows you to search the web for information related to the query."""
    tool = TavilySearch(
        max_results=5,
        topic="general",
        # include_answer=False,
        # include_raw_content=False,
        # include_images=False,
        # include_image_descriptions=False,
        # search_depth="basic",
        # time_range="day",
        # include_domains=None,
        # exclude_domains=None
    )
    return tool.invoke({"query": query})


Tavily API Key is set


## **EXA Search**

In [45]:
import getpass
import os

if os.environ.get("EXA_API_KEY"):
    print("Exa Search API Key is set")
else:
    os.environ["EXA_API_KEY"] = getpass.getpass("Exa API key:\n")

Exa Search API Key is set


In [46]:
from langchain_exa import ExaSearchResults

@tool
def exa_tool(query: str) -> str:
    """This tool allows you to search Exa for information related to the query."""
    # Initialize the ExaSearchResults tool
    search_tool = ExaSearchResults(exa_api_key=os.environ["EXA_API_KEY"])

    # Perform a search query
    search_results = search_tool._run(
        query="When was the last time the New York Knicks won the NBA Championship?",
        num_results=5,
        text_contents_options=True,
        highlights=True,
    )

    return search_results

# print("Search Results:", search_results)

## **Wikipedia Tools**

In [47]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia.wikipedia as wiki

@tool
def wiki_tool(query: str) -> str:
    """This tool allows you to search Wikipedia for information related to the query."""
    # Avoid anonymous/default UA rate-limiting responses that break JSON parsing.
    wiki.USER_AGENT = "langgraphexploration/0.1 (local learning notebook)"
    wiki.API_URL = "https://en.wikipedia.org/w/api.php"

    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)


## **Arxiv Tools**

In [48]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
import arxiv

@tool
def arxiv_tool(query: str) -> str:
    """This tool allows you to search Arxiv for papers related to the query."""
    query = "What are the latest papers on Large Language Models?"
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())

    try:
        # Works with wrapper/library combinations that still expose Search.results().
        result = arxiv_query.invoke(query)
    except AttributeError as ex:
        # Fallback for arxiv>=4 where Search.results() was removed.
        if "results" not in str(ex):
            raise

        client = arxiv.Client()
        search = arxiv.Search(
            query=query,
            max_results=3,
            sort_by=arxiv.SortCriterion.SubmittedDate,
        )
        papers = []
        for paper in client.results(search):
            authors = ", ".join(author.name for author in paper.authors[:5])
            papers.append(
                f"Published: {paper.published.date()}\n"
                f"Title: {paper.title}\n"
                f"Authors: {authors}\n"
                f"Summary: {paper.summary[:500]}"
            )

        result = "\n\n".join(papers) if papers else "No papers found."

    return result


## **DuckDuckGo search tool**

In [49]:
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def search_duckduckgo(query: str) -> str:
    """This tool allows you to search DuckDuckGo for information related to the query."""
    duckduckgo_search = DuckDuckGoSearchRun()
    return duckduckgo_search.invoke("Who is the president of USA?")

In [50]:
# search_duckduckgo.invoke("Who is the president of USA?")

In [51]:
# exa_tool.invoke("When was the last time the New York Knicks won the NBA Championship?")

## **Tools Binding**

In [ ]:
tools = [exa_tool, wiki_tool, arxiv_tool, search_duckduckgo, tavily_search]

llm_with_tools = llm.bind_tools(tools)
# response = llm_with_tools.invoke("What is the latest news on AI?")

# response.tool_calls

[{'name': 'tavily_search',
  'args': {'query': 'latest news on AI July 2026'},
  'id': 'call_AbJoqYgYUb6PQxKSGCZZQAEG',
  'type': 'tool_call'},
 {'name': 'search_duckduckgo',
  'args': {'query': 'latest AI news 2026'},
  'id': 'call_Zj2P8CuLb7ljywmG40pQATV7',
  'type': 'tool_call'},
 {'name': 'exa_tool',
  'args': {'query': 'latest developments in artificial intelligence 2026'},
  'id': 'call_2bNh4IeHcOgs6eFYGkheZ2Dr',
  'type': 'tool_call'},
 {'name': 'wiki_tool',
  'args': {'query': 'artificial intelligence latest news'},
  'id': 'call_VSFKYsTSjzVYSyiGVqkjBgz2',
  'type': 'tool_call'}]

## **LangGraph creation**

In [5]:
from typing import TypedDict, List

class graph_schema(TypedDict):
    messages: List


In [ ]:
def llm_node(state: graph_schema) -> graph_schema:
    messages = state['messages']
    
    # Prompt template for the LLM to use tools to answer questions
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant that can use tools to answer questions."),
            ("human", "{input}")
        ]
    )

    # LLM with tools to answer questions
    llm_with_tools

    chain = prompt | llm_with_tools

    response = chain.invoke({"input": messages})

    # update the state with the new message
    state['messages'] = messages + [response]

    return state